# Worked Python solutions
Use the same data folder as the paper. If running here inside SOLUTIONS, first set your working directory to the parent practice folder.

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm, binom, t
from scipy.stats import ttest_1samp, ttest_ind, ttest_rel, levene, shapiro, probplot
import statsmodels.formula.api as smf

## Q11
(a) ["Record_ID"]; (b) "coerce"; (c) 3; (d) 293

There are 304 raw rows and 301 unique Record_ID values. After retaining the first occurrence, eight distinct records fail the remaining required-value, validity or category checks. The original source also has similar-looking records with distinct IDs: do not remove them merely because other values agree.

In [ ]:
raw = pd.read_csv("data/cars_exam_raw.csv")
cars = raw.drop_duplicates(subset=["Record_ID"], keep="first").copy()
numeric_cols = ["Selling_Price", "Present_Price", "Kms_Driven"]
for col in numeric_cols:
    cars[col] = pd.to_numeric(cars[col], errors="coerce")
cars["Transmission"] = cars["Transmission"].str.strip().str.lower()
cars["Fuel_Type"] = cars["Fuel_Type"].str.strip().str.upper()
required = numeric_cols + ["Transmission", "Fuel_Type"]
cars = cars.dropna(subset=required)
cars = cars[(cars["Selling_Price"] > 0) &
            (cars["Present_Price"] > 0) &
            (cars["Kms_Driven"] >= 0)]
cars = cars[cars["Transmission"].isin(["automatic", "manual"]) &
            cars["Fuel_Type"].isin(["CNG", "DIESEL", "PETROL"])]
cars_clean = cars.reset_index(drop=True)

## Q12
(a) fillna; (b) 3.505

The temporary median is calculated before the validity filters. Assignment saves the returned Series. This exercise does not alter cars_clean.

In [ ]:
temp = raw.drop_duplicates(subset=["Record_ID"], keep="first").copy()
temp["Selling_Price"] = pd.to_numeric(temp["Selling_Price"], errors="coerce")
fill_value = temp["Selling_Price"].median()
temp["Selling_Price"] = temp["Selling_Price"].fillna(fill_value)

## Q13
(a) 3.500; (b) 5.144

Use the cleaned rows. The SD must use the sample convention (ddof=1).

In [ ]:
median_price = cars_clean["Selling_Price"].median()
sample_sd = cars_clean["Selling_Price"].std(ddof=1)

## Q14
(a) 0.8807; (b) False

Correlation describes linear association; it does not establish causation.

In [ ]:
r = cars_clean["Present_Price"].corr(cars_clean["Selling_Price"])

## Q29
(a) sf; (b) 4; (c) 0.2763

For a whole-number count, at least 5 is the same as greater than 4. SF excludes its cutoff.

In [ ]:
prob = binom.sf(4, 12, 0.3)

## Q30
(a) 12; (b) 0.7499; (c) 95.379

Use SD sqrt(144)=12 for scale; an interval probability subtracts CDF values and PPF uses cumulative probability.

In [ ]:
sigma = 12
prob = norm.cdf(90, loc=80, scale=sigma) - norm.cdf(60, loc=80, scale=sigma)
q90 = norm.ppf(0.90, loc=80, scale=sigma)

## Q31
(a) (5000, 25); (b) 1; (c) 2

Each row has 25 observations. Averaging across its columns uses axis=1, and the theoretical SD is 10/sqrt(25)=2.

In [ ]:
samples = np.random.normal(loc=70, scale=10, size=(5000, 25))
means = samples.mean(axis=1)

## Q32
(a) 0.667; (b) 0.569; (c) 0.764

There are 90 type-B observations and 60 successes. The interval SE uses the observed proportion. The approximation check n*p_hat*(1-p_hat) is 20.000.

In [ ]:
plants = pd.read_csv("data/gen_mod_plants.csv")
results_B = plants.loc[plants["mod.type"] == "B", "result"]
n = len(results_B)
p_hat = (results_B == "non-susceptible").mean()
se = np.sqrt(p_hat * (1 - p_hat) / n)
lower, upper = norm.interval(0.95, loc=p_hat, scale=se)

## Q33
(a) ttest_1samp; (b) 0.506; (c) 0.2181

The sample has 47 observations, so df=46. popmean must be the hypothesized 3.3, not the observed mean. T=-1.248786.

In [ ]:
weights = pd.read_csv("data/babyweights.csv")["weight"]
result = ttest_1samp(weights, popmean=3.3, alternative="two-sided")
s = weights.std(ddof=1)
p_value = result.pvalue

## Q34
(a) False; (b) 1.811; (c) 0.0749

Welch does not pool variances. Group sizes are 33 and 32; software gives df=62.684196.

In [ ]:
therapy = pd.read_csv("data/Anorexia.csv")
cog = therapy.loc[therapy["therapy"] == "cog", "change"]
cont = therapy.loc[therapy["therapy"] == "cont", "change"]
result = ttest_ind(cog, cont, equal_var=False, alternative="two-sided")

## Q35
(a) ttest_rel; (b) "less"; (c) -18.806; (d) -10.784

ttest_rel tests first argument minus second. The directional test result has a one-sided interval, so use the separate two-sided test on differences to obtain the requested two-sided 99% interval.

In [ ]:
chol = pd.read_csv("data/Cholesterol_level.csv")
d = chol["After"] - chol["Before"]
result = ttest_rel(chol["After"], chol["Before"], alternative="less")
ci = ttest_1samp(d, popmean=0, alternative="two-sided").confidence_interval(0.99)

## Q36
(a) 22.50; (b) 3.162; (c) 0.0008; (d) Yes

Use p0=0.5 in the test SE and condition; do not substitute the CI SE from Q32.

In [ ]:
p0 = 0.5
z = (p_hat - p0) / np.sqrt(p0 * (1 - p0) / n)
p_value = norm.sf(z)

## Q37
(a) 0.7167; (b) 0.5184; (c) 0.7757; (d) 291

The intercept is included by default. With 293 fitted rows, SLR estimates two coefficients and has 291 residual df. All price variables here are in the original dataset scale.

In [ ]:
m1 = smf.ols("Selling_Price ~ Present_Price", data=cars_clean).fit()
print(m1.params)
print(m1.rsquared, m1.df_resid)

## Q38
(a) C; (b) -1.4357; (c) 3; (d) 0.8041

The manual coefficient is an adjusted mean difference holding Present_Price and Kms_Driven fixed. There are three nonintercept coefficient columns; the intercept does not count toward numerator df.

In [ ]:
formula = ("Selling_Price ~ Present_Price + Kms_Driven + "
           "C(Transmission, Treatment('automatic'))")
m2 = smf.ols(formula, data=cars_clean).fit()
print(m2.params)
print(m2.df_model, m2.rsquared_adj)

## Q39
(a) 0.6129; (b) 0.4549; (c) -0.7714; (d) 5.5356

With automatic as reference, manual slope adds the interaction coefficient. The group difference at x is the manual indicator coefficient plus x times the interaction coefficient; prediction adds the intercept and the relevant slope contribution.

In [ ]:
mi = smf.ols(
    "Selling_Price ~ Present_Price * C(Transmission, Treatment('automatic'))",
    data=cars_clean
).fit()
new = pd.DataFrame({"Present_Price": [10, 11, 10, 11],
                    "Transmission": ["automatic", "automatic", "manual", "manual"]})
p = mi.predict(new)
auto_slope = p.iloc[1] - p.iloc[0]
manual_slope = p.iloc[3] - p.iloc[2]
gap_at_10 = p.iloc[2] - p.iloc[0]
manual_at_10 = p.iloc[2]

## Q40
(a) 6; (b) 1; (c) I(Present_Price**2); (d) 0.6065

Use standardized rather than raw residuals. Cook and SR thresholds flag observations for investigation. For b0+b1*x+b2*x^2, the exact 10-to-11 change is b1+21*b2, not the derivative at x=10.

In [ ]:
influence = m1.get_influence()
sr = influence.resid_studentized_internal
cooks = influence.cooks_distance[0]
mq = smf.ols("Selling_Price ~ Present_Price + I(Present_Price**2)", data=cars_clean).fit()
outlier_count = np.sum(np.abs(sr) > 3)
influential_count = np.sum(cooks > 1)
qpred = mq.predict(pd.DataFrame({"Present_Price": [10, 11]}))
change_10_to_11 = qpred.iloc[1] - qpred.iloc[0]